# Quick SLM — 01 · Data preparation

Streams the four pretraining corpora from HuggingFace, tokenizes them into per-source
`uint16` `.bin` files, and interleaves them at CTX-window granularity into `combined.bin`.

| Source | Filter | Share | Tokens |
|---|---|---:|---:|
| FineWeb-Edu | `int_score >= 4` | 77.5% | 7.75 B |
| FineMath | `int_score == 5` within `finemath-4plus` | 10% | 1.00 B |
| Tool-calling | xLAM x3 epochs, then Glaive | 2.5% | 0.25 B |
| StarCoder | Python subset | 10% | 1.00 B |

Every step is **resumable**. Bulk writes land on local Colab disk; Drive only ever receives
the manifest, the tokenizer, and size-verified copies of finished `.bin` files, because
Drive's FUSE mount silently truncates large writes under load.

All logic lives in `quick_slm_trainer.pretraining`. This notebook is the runner.

## 1 · Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Framework

In [ ]:
EXTRAS = "data"

# Framework and data live on Drive (no GitHub). Upload the repo folder
# (with pyproject.toml, README.md, and framework/) into DRIVE_ROOT/code once.
# Every notebook installs it from there.
DRIVE_ROOT = "/content/drive/MyDrive/quick-slm"

import subprocess, sys
from pathlib import Path

root = Path(DRIVE_ROOT)
candidates = [root / "code", root, root / "quick-slm"]
REPO_DIR = next((p for p in candidates if (p / "pyproject.toml").exists()), None)
if REPO_DIR is None:
    looked = "\n  ".join(str(p) for p in candidates)
    raise RuntimeError(
        "No pyproject.toml found on Drive. Upload the repo (pyproject.toml, "
        f"README.md, and framework/) to {root / 'code'}, then re-run.\nLooked in:\n  "
        + looked
    )

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"], check=True)

import v1.quick_slm_trainer as q
print("quick-slm-trainer", q.__version__, "from", Path(q.__file__).parent)
# Refuse to run outside the support window this training version declares in
# training/v1/framework.json. A framework newer than v1's window would not fail
# loudly, it would build a different corpus and the difference would surface as
# unexplained numbers weeks later. If this raises, install the archived build it
# names rather than editing this cell.
if hasattr(q, "require_framework"):
    q.require_framework("v1", REPO_DIR)
else:
    raise RuntimeError(
        "quick-slm-trainer " + q.__version__ + " predates the support-window check; "
        "training v1 requires >=1.0. See SUPPORT.md."
    )


## 3 · Layout, config, tokenizer

`Layout` is the only place a directory is named. `DataConfig` carries the budgets and
the filter thresholds the paper quotes. The tokenizer is LLaMA-2 plus the fourteen
reserved domain tokens, built once and reused by every later stage.

In [ ]:
from v1.quick_slm_trainer import Layout, Manifest, pretrain_v1
from v1.quick_slm_trainer.tokenizer import EXPECTED_VOCAB_SIZE, get_or_build_tokenizer

layout = Layout().mkdirs()
cfg = pretrain_v1()

tok = get_or_build_tokenizer(cfg.data.tokenizer_repo, layout.tokenizer_dir)
assert len(tok) == EXPECTED_VOCAB_SIZE, f'{len(tok)} != {EXPECTED_VOCAB_SIZE}'

manifest = Manifest(layout, cfg.data.budgets)
manifest.reconcile_all()

print(f'vocab      : {len(tok):,}')
print(f'eos id     : {tok.eos_token_id}')
print(f'ctx        : {cfg.data.ctx:,}')
print(f'target     : {cfg.data.target_total_tokens:,} tokens')
for k, share in cfg.data.shares().items():
    print(f'  {k:<14s} {cfg.data.budgets[k]:>15,}  ({100 * share:.1f}%)')

## 4 · Stream the sources

Each call resumes from the manifest, tokenizes until its budget is met, fsyncs every
50 batches, and copies the finished file to Drive with a byte-count check.

Run the cell as many times as it takes. FineWeb-Edu is the long one, roughly three days
of streaming at the tier-4 hit rate.

In [ ]:
from v1.quick_slm_trainer.pretraining import build_source, stream_source

for key in cfg.data.budgets:
    source = build_source(key, cfg.data)
    stream_source(
        layout=layout, manifest=manifest, cfg=cfg.data, tok=tok,
        key=key, documents=source.documents, text_fn=source.text_fn,
        budget=cfg.data.budgets[key],
    )

### Integrity check

Manifest claims against what is actually on local disk and on Drive.

In [ ]:
report, ok = manifest.integrity_report()
print(report)
print('\nOK' if ok else '\nMISMATCH — do not combine until this is resolved')

## 5 · Combine

Windows are drawn from the four sources in a shuffled pattern, so any contiguous slice of
`combined.bin` matches the target ratio and the trainer never needs to know about source
boundaries.

`combine_mode='strict'` refuses to run while any source is short of budget. A corpus whose
ratios are not the ratios the paper claims is worse than no corpus. To proceed anyway, set
`cfg.data.combine_mode` to `'cap_proportional'` (scales every source down, preserving the
77.5 / 10 / 2.5 / 10 ratio) or `'use_all'` (takes what exists, skewing the ratio).

In [ ]:
from v1.quick_slm_trainer.pretraining import combine_sources

total = combine_sources(layout=layout, manifest=manifest, cfg=cfg.data)
manifest.finalise(tokenizer_dir=layout.tokenizer_dir, ctx=cfg.data.ctx, vocab_size=len(tok))
cfg.save(layout.data_dir / 'pretrain_config.json')

print(f'\ncombined.bin: {total:,} tokens ({total // cfg.data.ctx:,} windows)')
print(f'manifest    : {layout.manifest_path}')